# Day 8 lab: Best practices, and late data in streaming

**Goal:** Improve a slow, messy job one change at a time with a measured reason for each change, and handle late events in a stream.

**Time:** about 2 h 30 min. Checklist: 6.1, 6.3–6.9.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Chapter 5: Unleashing UDFs — 20 min

Type the Python UDF, pandas UDF and Arrow UDF sections of the official guide (skip UDTFs). For each, run `explain()` and find the evaluation node (`ArrowEvalPython` or `BatchEvalPython`), then write the same logic with built-in functions and compare: the built-in version has no Python node and stays inside whole-stage codegen.

[Chapter 5: Unleashing UDFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html) · [Arrow Python UDFs](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_python_udf.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/user_guide/udfandudtf.ipynb)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. Testing PySpark — 20 min

Type the official example application (`remove_extra_spaces`) and testing options 1 and 3: `assertDataFrameEqual`, `assertSchemaEqual`, and a pytest `spark` fixture. Run the tests with `pytest -q`. This is how real projects keep Spark code safe to refactor.

[Testing PySpark](https://spark.apache.org/docs/latest/api/python/getting_started/testing_pyspark.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/getting_started/testing_pyspark.ipynb)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: refactor bad_job.py — 40 min

Download `bad_job.py` next to `lab_utils.py`, run it 3 times and note the median time. Copy it to `good_job.py` and apply one fix per run, recording the time and the plan change: (1) both UDFs → `F.coalesce(F.upper(F.trim("status")), F.lit("UNKNOWN"))` and a plain `F.col("amount") > MIN_AMOUNT` filter; (2) the 40-step `withColumn` loop → one `select("*", *[F.lit(i).alias(f"flag_{i}") for i in range(40)])`; (3) stop disabling broadcasts, or use `F.broadcast(customers)`; (4) filter before the join; (5) write in parallel instead of `coalesce(1)`; (6) move paths and thresholds into a JSON config read through `argparse`. Put the logic in small functions (`clean_status`, `add_flags`, `build_report`) so Exercise 2 can test them. 

**Check:** about 10 s down to about 4 s on this module's data, with identical output: `assertDataFrameEqual(bad.select(sorted(bad.columns)), good.select(sorted(good.columns)))`. Proves 6.1, 6.5, 6.7, 6.8, 6.2.

[Download bad_job.py](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/bad_job.py) · [Solution: good_job.py](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/good_job.py) · [Solution: good_job.json](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/good_job.json)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: test it like a real project — 25 min

Write `test_good_job.py` with a session-scoped pytest `spark` fixture, as in the Testing PySpark guide, and tests for `clean_status` (padded, lower-case and null statuses become `PAID`, `NEW`, `UNKNOWN`), `add_flags` (the column names) and `build_report` on a 3-row orders table (only big orders from VIP countries survive). Compare with small expected DataFrames through `assertDataFrameEqual`. 

**Check:** `pytest -q` passes; then break `clean_status` on purpose (drop the `trim`) and read the diff the assertion prints. Proves 6.1, 6.7, 6.8.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/test_good_job.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. Star schema and dynamic partition pruning — 20 min

Write orders with `partitionBy("order_date")`. Build a small date dimension: `spark.range(30).select(F.date_add(F.lit("2026-01-01"), F.col("id").cast("int")).alias("order_date"), (F.col("id") % 7).alias("dow"))`. Join it with the orders and filter `dow = 3`. Find `dynamicpruningexpression` in the fact scan's `PartitionFilters`, and in the SQL tab check that only about a seventh of the files were read. Proves 6.4.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Late data with a watermark — 25 min

Stream `spark.readStream.schema("ts timestamp, user string").json("events_in").withWatermark("ts", "10 minutes").groupBy(F.window("ts", "5 minutes")).count()` to Parquet in append mode with a checkpoint location and `trigger(availableNow=True)`, then read the output. Drop one JSON-lines file at a time into `events_in/` and rerun after each: (1) events at 12:00, 12:01, 12:06, 12:09 → nothing emitted, watermark 11:59; (2) one event at 12:31 → windows 12:00 and 12:05 emitted with 2 rows each, watermark 12:21; (3) a late event at 12:07 → dropped, counts unchanged; (4) an event at 12:45 → the 12:20 and 12:30 windows appear. Print `query.lastProgress["eventTime"]` each time. Use `spark.conf.set("spark.sql.session.timeZone", "UTC")` so the times read as written. Proves 6.9.

[Structured Streaming quick example](https://spark.apache.org/docs/latest/streaming/getting-started.html) · [Official windowed word count](https://github.com/apache/spark/blob/v4.2.0/examples/src/main/python/sql/streaming/structured_network_wordcount_windowed.py)

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*